# DNS Anycast, servidores y distribucion de carga

Este notebook responde:

1. Como la topologia ayuda al analisis futuro de DNS Anycast en Chile.
2. Que paises requeririan servidor de nube para reducir latencia (aprox. por topologia).
3. A que ASN conviene conectar un servidor para minimizar atraccion de trafico internacional.
4. Como mejorar distribucion de carga.
5. A que ASN conviene conectar para alcanzar la mayor parte de redes del pais.

Nota: se usa heuristica de pais por nombre de ASN; no reemplaza datos de RTT reales.


In [1]:
from pathlib import Path
import re
import pandas as pd
import numpy as np
from IPython.display import display, Markdown

DATA_DIR = Path('../data/csv') if Path('../data/csv').exists() else Path('data/csv')

nodes = pd.read_csv(DATA_DIR / 'merged' / 'nodes.csv')
edges = pd.read_csv(DATA_DIR / 'merged' / 'edges.csv')
nodes['name'] = nodes['name'].fillna('')
nodes['degree'] = nodes['in_degree'] + nodes['out_degree']

id2asn = dict(zip(nodes['node_id'], nodes['asn']))
asn2name = dict(zip(nodes['asn'], nodes['name']))
print('Merged:', len(nodes), 'nodes,', len(edges), 'edges')

Merged: 9839 nodes, 29649 edges


In [2]:
def infer_chile_asns(nodes_df):
    explicit = set(nodes_df.loc[nodes_df['name'].str.contains(r'\bchile\b', case=False, regex=True), 'asn'].astype(int))
    manual = {27986, 6568, 27925, 27651, 22047, 52341, 18822, 14117, 10834, 20015, 6471, 6429, 14259, 27678, 20191, 23140, 64112}
    return explicit | manual


def infer_country(name):
    n = (name or '').lower()
    patterns = {
        'Chile': [r'\bchile\b', r'santiago', r'vtr'],
        'Argentina': [r'argentina', r'buenos aires'],
        'Peru': [r'peru', r'per\u00fa', r'lima'],
        'Brazil': [r'brazil', r'brasil'],
        'Colombia': [r'colombia'],
        'Mexico': [r'mexico', r'm\u00e9xico'],
        'United States': [r'united states', r'\binc\.?\b', r'\bllc\b', r'virginia', r'new york'],
        'Germany': [r'gmbh', r'germany'],
        'Italy': [r'italia', r'italy'],
        'Spain': [r'espa\u00f1a', r'spain'],
        'Netherlands': [r'netherlands', r'\bb\.v\.\b'],
    }
    for c, pats in patterns.items():
        for p in pats:
            if re.search(p, n):
                return c
    return 'Unknown/Multinational'


def norm(s):
    return s / max(s.max(), 1)

In [3]:
chile_asn = infer_chile_asns(nodes)

from collections import defaultdict
out_neighbors = defaultdict(set)
out_weight = defaultdict(float)

for s, d, w in edges[['src_id', 'dst_id', 'weight']].itertuples(index=False):
    a = int(id2asn[s]); b = int(id2asn[d])
    out_neighbors[a].add(b)
    out_weight[(a, b)] += float(w)

rows = []
for a in sorted(chile_asn):
    neigh = out_neighbors.get(a, set())
    if not neigh:
        continue
    dom = [b for b in neigh if b in chile_asn]
    intl = [b for b in neigh if b not in chile_asn]
    dom_w = sum(out_weight[(a, b)] for b in dom)
    intl_w = sum(out_weight[(a, b)] for b in intl)
    total_w = dom_w + intl_w
    rows.append({
        'asn': a,
        'name': asn2name.get(a, ''),
        'out_neighbors': len(neigh),
        'domestic_neighbors': len(dom),
        'foreign_neighbors': len(intl),
        'domestic_weight': dom_w,
        'foreign_weight': intl_w,
        'domestic_weight_ratio': (dom_w / total_w) if total_w > 0 else 0.0,
        'path_occurrences': int(nodes.loc[nodes['asn'] == a, 'path_occurrences'].max()) if (nodes['asn'] == a).any() else 0,
    })

attach = pd.DataFrame(rows)
attach['anycast_score'] = 0.45 * norm(attach['domestic_neighbors']) + 0.35 * attach['domestic_weight_ratio'] + 0.20 * norm(attach['path_occurrences'])
display(attach.sort_values('anycast_score', ascending=False).head(20))

,asn,name,out_neighbors,domestic_neighbors,foreign_neighbors,domestic_weight,foreign_weight,domestic_weight_ratio,path_occurrences,anycast_score
5,14259,GTD Chile,50,6,44,23718.0,1629.0,0.935732,54976,0.955281
13,27986,entel IP Internacional,3,3,0,30792.0,0.0,1.000000,61849,0.775000
17,263237,PowerHost Chile,26,3,23,13711.0,7842.0,0.636153,43825,0.589370
6,18822,Gtd Manquehue,1,1,0,2.0,0.0,1.000000,10442,0.458766
7,19338,Telmex Chile Internet,1,1,0,10.0,0.0,1.000000,20,0.425065
11,27651,entel MPLS,2,1,1,2.0,2.0,0.500000,25695,0.333089
9,20191,Pontificia Universidad Catolica de Chile,2,1,1,1.0,1.0,0.500000,527,0.251704
19,266803,TIC CHILE COMUNICACIONES,2,1,1,20.0,20.0,0.500000,116,0.250375
20,273148,SMARTNET CHILE,2,1,1,1.0,1.0,0.500000,22,0.250071
4,14117,Telefonica del Sur,7,1,6,2.0,900.0,0.002217,13952,0.120892


## 1) Topologia y analisis futuro DNS Anycast

In [4]:
top_anycast = attach.sort_values('anycast_score', ascending=False).head(10)
display(top_anycast[['asn', 'name', 'domestic_neighbors', 'domestic_weight_ratio', 'path_occurrences', 'anycast_score']])

lines = [
    '- El ranking combina cobertura domestica, riesgo de fuga internacional y centralidad.',
    '- Candidatos Anycast iniciales: ' + ', '.join('AS' + str(int(a)) for a in top_anycast['asn'].head(3)),
]
display(Markdown("\n".join(lines)))

,asn,name,domestic_neighbors,domestic_weight_ratio,path_occurrences,anycast_score
5,14259,GTD Chile,6,0.935732,54976,0.955281
13,27986,entel IP Internacional,3,1.000000,61849,0.775000
17,263237,PowerHost Chile,3,0.636153,43825,0.589370
6,18822,Gtd Manquehue,1,1.000000,10442,0.458766
7,19338,Telmex Chile Internet,1,1.000000,20,0.425065
11,27651,entel MPLS,1,0.500000,25695,0.333089
9,20191,Pontificia Universidad Catolica de Chile,1,0.500000,527,0.251704
19,266803,TIC CHILE COMUNICACIONES,1,0.500000,116,0.250375
20,273148,SMARTNET CHILE,1,0.500000,22,0.250071
4,14117,Telefonica del Sur,1,0.002217,13952,0.120892


- El ranking combina cobertura domestica, riesgo de fuga internacional y centralidad.
- Candidatos Anycast iniciales: AS14259, AS27986, AS263237

## 2) Paises que podrian requerir servidor de nube para reducir latencia (aprox.)

In [5]:
rows_country = []
for s, d, w in edges[['src_id', 'dst_id', 'weight']].itertuples(index=False):
    src = int(id2asn[s]); dst = int(id2asn[d])
    if src not in chile_asn:
        continue
    c = infer_country(asn2name.get(dst, ''))
    if c == 'Chile':
        continue
    rows_country.append((c, dst, float(w)))

country_df = pd.DataFrame(rows_country, columns=['country', 'dst_asn', 'weight'])
if not country_df.empty:
    country_priority = country_df.groupby('country', as_index=False).agg(edge_weight_sum=('weight', 'sum'), unique_asns=('dst_asn', 'nunique')).sort_values('edge_weight_sum', ascending=False)
else:
    country_priority = pd.DataFrame(columns=['country', 'edge_weight_sum', 'unique_asns'])

display(country_priority.head(15))
known = country_priority[~country_priority['country'].isin(['Unknown/Multinational'])].head(5)
if len(known):
    display(Markdown('Prioridad inicial: ' + ', '.join(known['country'].tolist())))
else:
    display(Markdown('Sin metadatos geograficos adicionales, no hay priorizacion robusta por pais.'))

,country,edge_weight_sum,unique_asns
2,Unknown/Multinational,83957.0,120
0,Argentina,1131.0,2
1,United States,8.0,1


Prioridad inicial: Argentina, United States

## 3) ASN para minimizar atraccion de trafico de otro pais

In [6]:
low_leak = attach[attach['out_neighbors'] >= 3].sort_values(['domestic_weight_ratio', 'domestic_neighbors', 'path_occurrences'], ascending=[False, False, False])
display(low_leak[['asn', 'name', 'out_neighbors', 'domestic_neighbors', 'domestic_weight_ratio', 'path_occurrences']].head(10))
if len(low_leak):
    b = low_leak.iloc[0]
    display(Markdown(f"Candidato principal anti-leak: AS{int(b['asn'])} ({b['name']}), ratio domestico={b['domestic_weight_ratio']:.3f}."))

,asn,name,out_neighbors,domestic_neighbors,domestic_weight_ratio,path_occurrences
13,27986,entel IP Internacional,3,3,1.000000,61849
5,14259,GTD Chile,50,6,0.935732,54976
17,263237,PowerHost Chile,26,3,0.636153,43825
12,27678,NIC Chile,4,1,0.038462,240
1,6471,entel Chile Servicios Fijos,22,1,0.023748,6842
16,64112,PIT Chile - Transit,6,1,0.023529,353
4,14117,Telefonica del Sur,7,1,0.002217,13952
0,6429,CLARO CHILE AS6429,18,1,0.001768,7093
3,10834,Telefonica Empresas,5,0,0.000000,5257


Candidato principal anti-leak: AS27986 (entel IP Internacional), ratio domestico=1.000.

## 4) Mejor distribucion de carga

In [7]:
reach = attach.sort_values(['domestic_neighbors', 'path_occurrences', 'domestic_weight_ratio'], ascending=False).head(5)
bal = attach[(attach['out_neighbors'] >= 3) & (attach['domestic_weight_ratio'] >= 0.35) & (attach['domestic_weight_ratio'] <= 0.85)]      .sort_values(['domestic_neighbors', 'path_occurrences'], ascending=False)
if bal.empty:
    temp = attach.copy()
    temp['dist_50'] = (temp['domestic_weight_ratio'] - 0.5).abs()
    bal = temp.sort_values(['dist_50', 'domestic_neighbors', 'path_occurrences']).head(3)

display(Markdown('Nodos de mayor alcance domestico:'))
display(reach[['asn', 'name', 'domestic_neighbors', 'domestic_weight_ratio', 'path_occurrences']])

display(Markdown('Nodos para balanceo:'))
display(bal[['asn', 'name', 'domestic_neighbors', 'domestic_weight_ratio', 'path_occurrences']].head(5))

display(Markdown('Recomendacion: multihoming Anycast en al menos 2 ASNs con perfil distinto y ajuste de communities/local-pref/prepend.'))

Nodos de mayor alcance domestico:

,asn,name,domestic_neighbors,domestic_weight_ratio,path_occurrences
5,14259,GTD Chile,6,0.935732,54976
13,27986,entel IP Internacional,3,1.000000,61849
17,263237,PowerHost Chile,3,0.636153,43825
11,27651,entel MPLS,1,0.500000,25695
4,14117,Telefonica del Sur,1,0.002217,13952


Nodos para balanceo:

,asn,name,domestic_neighbors,domestic_weight_ratio,path_occurrences
17,263237,PowerHost Chile,3,0.636153,43825


Recomendacion: multihoming Anycast en al menos 2 ASNs con perfil distinto y ajuste de communities/local-pref/prepend.

## 5) ASN para alcanzar mayoria de redes dentro del pais

In [8]:
reach_rank = attach.sort_values(['domestic_neighbors', 'domestic_weight', 'path_occurrences'], ascending=False)
display(reach_rank[['asn', 'name', 'domestic_neighbors', 'domestic_weight', 'out_neighbors', 'path_occurrences']].head(10))
if len(reach_rank):
    t = reach_rank.iloc[0]
    display(Markdown(f"Candidato principal por alcance nacional: AS{int(t['asn'])} ({t['name']})."))

,asn,name,domestic_neighbors,domestic_weight,out_neighbors,path_occurrences
5,14259,GTD Chile,6,23718.0,50,54976
13,27986,entel IP Internacional,3,30792.0,3,61849
17,263237,PowerHost Chile,3,13711.0,26,43825
1,6471,entel Chile Servicios Fijos,1,55.0,22,6842
19,266803,TIC CHILE COMUNICACIONES,1,20.0,2,116
7,19338,Telmex Chile Internet,1,10.0,1,20
11,27651,entel MPLS,1,2.0,2,25695
4,14117,Telefonica del Sur,1,2.0,7,13952
6,18822,Gtd Manquehue,1,2.0,1,10442
0,6429,CLARO CHILE AS6429,1,2.0,18,7093


Candidato principal por alcance nacional: AS14259 (GTD Chile).

## Advertencias

- `weight` no es RTT.
- La inferencia de pais por nombre es solo una aproximacion.
- Para decisiones de produccion, sumar RTT RIPE Atlas + geolocalizacion ASN confiable.
